# Study & Task Planner Agent

A **Practical Assistant Agent** that helps a student manage a to-do list, check the weather and look up topics on Wikipedia. It talks to two real external services (**Open-Meteo** for weather, **Wikipedia** for summaries), remembers the conversation (memory), and **stops to ask the human for approval** before any destructive action (deleting one task or clearing the whole list). Built with LangGraph and Google Gemini.

Files in the repo: `agent.py` (same code as this notebook), `main.py` (terminal chat), `app.py` (Streamlit chat), `README.md`.

## 1. Setup

Install the libraries, then add your key. **In Google Colab:** click the key icon (Secrets), add a secret named `GOOGLE_API_KEY`, and turn on notebook access. The key below is intentionally left empty in the code.

In [ ]:
!pip install -q langchain langchain-core langgraph langchain-google-genai requests

In [ ]:
import os
from getpass import getpass

# GOOGLE_API_KEY=""   <- keep it empty in code you upload!
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    if not os.getenv("GOOGLE_API_KEY"):
        os.environ["GOOGLE_API_KEY"] = getpass("Paste your Gemini API key: ")

## 2. Tools

Every tool is a Python function with `@tool` and a docstring (the AI reads the docstring to know when to use it).

- **Custom tools** (no API): `add_task`, `list_tasks`, `complete_task`, `delete_task` (destructive), `clear_all_tasks` (destructive)
- **External APIs**: `get_weather` (Open-Meteo), `search_wikipedia` (Wikipedia)

In [ ]:
import os
from typing import Annotated, TypedDict

import requests
from langchain_core.messages import AIMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.types import interrupt

In [ ]:
# The task list lives in normal Python memory (a list of dictionaries).
TASKS = []
_next_id = 1


@tool
def add_task(title: str) -> str:
    """Add a new task to the to-do list. Use this when the user wants to remember or plan something."""
    global _next_id
    TASKS.append({"id": _next_id, "title": title, "done": False})
    _next_id += 1
    return f"Task added: #{_next_id - 1} '{title}'"


@tool
def list_tasks() -> str:
    """Show all tasks in the to-do list with their id and whether they are done."""
    if not TASKS:
        return "The task list is empty."
    lines = [f"#{t['id']} [{'x' if t['done'] else ' '}] {t['title']}" for t in TASKS]
    return "\n".join(lines)


@tool
def complete_task(task_id: int) -> str:
    """Mark one task as done, using its id number (see list_tasks)."""
    for t in TASKS:
        if t["id"] == task_id:
            t["done"] = True
            return f"Task #{task_id} marked as done."
    return f"No task with id {task_id}."


@tool
def delete_task(task_id: int) -> str:
    """Permanently delete one task by its id number. DESTRUCTIVE: needs human approval."""
    for t in TASKS:
        if t["id"] == task_id:
            TASKS.remove(t)
            return f"Task #{task_id} deleted."
    return f"No task with id {task_id}."


@tool
def clear_all_tasks() -> str:
    """Permanently delete ALL tasks from the list. DESTRUCTIVE: needs human approval."""
    count = len(TASKS)
    TASKS.clear()
    return f"All {count} tasks deleted."

In [ ]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city (uses the free Open-Meteo API)."""
    try:
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1},
            timeout=10,
        ).json()
        if not geo.get("results"):
            return f"Could not find a city called '{city}'."
        place = geo["results"][0]
        weather = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": "temperature_2m,wind_speed_10m,precipitation",
            },
            timeout=10,
        ).json()["current"]
        return (
            f"Weather in {place['name']}, {place.get('country', '')}: "
            f"{weather['temperature_2m']} C, wind {weather['wind_speed_10m']} km/h, "
            f"precipitation {weather['precipitation']} mm."
        )
    except Exception as error:  # network problems, bad response, etc.
        return f"Weather service error: {error}"


@tool
def search_wikipedia(topic: str) -> str:
    """Search Wikipedia and return a short summary of a topic (free Wikipedia API)."""
    try:
        headers = {"User-Agent": "StudyPlannerAgent/1.0 (student project)"}
        found = requests.get(
            "https://en.wikipedia.org/w/api.php",
            params={"action": "opensearch", "search": topic, "limit": 1, "format": "json"},
            headers=headers,
            timeout=10,
        ).json()
        if not found[1]:
            return f"No Wikipedia page found for '{topic}'."
        title = found[1][0]
        summary = requests.get(
            "https://en.wikipedia.org/api/rest_v1/page/summary/" + title.replace(" ", "_"),
            headers=headers,
            timeout=10,
        ).json()
        return f"{summary['title']}: {summary['extract']}"
    except Exception as error:
        return f"Wikipedia error: {error}"


SAFE_TOOLS = [add_task, list_tasks, complete_task, get_weather, search_wikipedia]
DESTRUCTIVE_TOOLS = [delete_task, clear_all_tasks]
ALL_TOOLS = SAFE_TOOLS + DESTRUCTIVE_TOOLS
DESTRUCTIVE_NAMES = {t.name for t in DESTRUCTIVE_TOOLS}
TOOLS_BY_NAME = {t.name: t for t in ALL_TOOLS}

## 3. State + Graph

- **State**: just the list of messages (the conversation).
- **Nodes**: `assistant` (the AI thinks), `safe_tools` (runs harmless tools), `approval` (pauses and asks the human before destructive tools).
- **Memory**: `MemorySaver` checkpointer + a `thread_id`.

In [ ]:
SYSTEM_PROMPT = (
    "You are a friendly Study & Task Planner assistant. You manage a to-do list, "
    "check the weather and look things up on Wikipedia. "
    "When the user asks to delete tasks, call the delete tool directly - the system "
    "will ask the user for approval, so do NOT ask for confirmation yourself. "
    "If a tool says an action was cancelled, tell the user nothing was changed."
)


class State(TypedDict):
    # 'messages' is the whole conversation. add_messages appends new messages to it.
    messages: Annotated[list, add_messages]


def build_graph(llm, checkpointer=None):
    """Build the agent graph. 'llm' is the chat model, 'checkpointer' stores memory."""
    llm_with_tools = llm.bind_tools(ALL_TOOLS)

    def assistant(state: State):
        """The AI reads the conversation and either answers or asks for a tool call."""
        reply = llm_with_tools.invoke([SystemMessage(content=SYSTEM_PROMPT)] + state["messages"])
        return {"messages": [reply]}

    def route(state: State):
        """Decide the next step: finish, run safe tools, or ask for approval first."""
        last = state["messages"][-1]
        calls = getattr(last, "tool_calls", None)
        if not calls:
            return END
        if any(c["name"] in DESTRUCTIVE_NAMES for c in calls):
            return "approval"
        return "safe_tools"

    def approval(state: State):
        """Pause and ask the human before any destructive tool runs."""
        calls = state["messages"][-1].tool_calls

        # Step 1: ask the human about every destructive call.
        # (interrupt() pauses the graph; the node restarts when the human answers,
        #  so nothing with side effects may happen before this loop.)
        approved = {}
        for call in calls:
            if call["name"] in DESTRUCTIVE_NAMES:
                answer = interrupt(
                    {
                        "action": call["name"],
                        "args": call["args"],
                        "question": describe_action(call),
                    }
                )
                approved[call["id"]] = str(answer).strip().lower() in ("yes", "y", "approve", "ok")

        # Step 2: run the allowed calls, skip the denied ones.
        results = []
        for call in calls:
            if call["name"] in DESTRUCTIVE_NAMES and not approved[call["id"]]:
                text = "Action cancelled: the user denied approval. Nothing was changed."
            else:
                text = TOOLS_BY_NAME[call["name"]].invoke(call["args"])
            results.append(ToolMessage(content=str(text), tool_call_id=call["id"], name=call["name"]))
        return {"messages": results}

    builder = StateGraph(State)
    builder.add_node("assistant", assistant)
    builder.add_node("safe_tools", ToolNode(SAFE_TOOLS))
    builder.add_node("approval", approval)
    builder.add_edge(START, "assistant")
    builder.add_conditional_edges("assistant", route, ["safe_tools", "approval", END])
    builder.add_edge("safe_tools", "assistant")
    builder.add_edge("approval", "assistant")
    return builder.compile(checkpointer=checkpointer)


def describe_action(call) -> str:
    """Human-readable text for the approval question."""
    if call["name"] == "delete_task":
        task_id = call["args"].get("task_id")
        title = next((t["title"] for t in TASKS if t["id"] == task_id), "unknown task")
        return f"Delete task #{task_id} ('{title}')?"
    if call["name"] == "clear_all_tasks":
        return f"Delete ALL {len(TASKS)} tasks?"
    return f"Run {call['name']}?"

In [ ]:
def make_llm():
    """Gemini chat model. Reads GOOGLE_API_KEY from the environment."""
    from langchain_google_genai import ChatGoogleGenerativeAI

    # If this model name is rejected, open Google AI Studio and copy a current free model name.
    return ChatGoogleGenerativeAI(model=os.getenv("GEMINI_MODEL", "gemini-2.5-flash"), temperature=0)


def create_agent(llm=None):
    """Create the agent with in-memory conversation memory."""
    return build_graph(llm or make_llm(), MemorySaver())

In [ ]:
def message_text(message) -> str:
    """Get plain text out of a message (Gemini sometimes returns a list of parts)."""
    content = message.content
    if isinstance(content, list):
        return "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in content)
    return str(content)


def pending_approval(agent, config):
    """If the agent is paused waiting for approval, return the question info, else None."""
    state = agent.get_state(config)
    if state.next and state.tasks and state.tasks[0].interrupts:
        return state.tasks[0].interrupts[0].value
    return None


def chat_turn(agent, user_text, thread_id):
    """Send one user message. Returns ('reply', text) or ('approval', info)."""
    config = {"configurable": {"thread_id": thread_id}}
    result = agent.invoke({"messages": [("user", user_text)]}, config)
    return _outcome(agent, result, config)


def resume_turn(agent, decision, thread_id):
    """Answer a pending approval question with 'yes' or 'no'."""
    from langgraph.types import Command

    config = {"configurable": {"thread_id": thread_id}}
    result = agent.invoke(Command(resume=decision), config)
    return _outcome(agent, result, config)


def _outcome(agent, result, config):
    info = pending_approval(agent, config)
    if info:
        return "approval", info
    last_ai = next((m for m in reversed(result["messages"]) if isinstance(m, AIMessage)), None)
    return "reply", message_text(last_ai) if last_ai else ""

In [ ]:
agent = create_agent()
print('Agent ready')

### Graph diagram

In [ ]:
from IPython.display import Image, display

try:
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception as e:  # the PNG needs internet access to mermaid.ink
    print("Could not draw PNG:", e)
    print(agent.get_graph().draw_mermaid())

## 4. Test queries

Each test is in its own cell. `thread_id` is the conversation id: the same id means the agent remembers earlier messages.

In [ ]:
def ask(text, thread_id="test-1"):
    """Send a message and print the full trajectory (every step the agent took)."""
    config = {"configurable": {"thread_id": thread_id}}
    result = agent.invoke({"messages": [("user", text)]}, config)
    for m in result["messages"]:
        m.pretty_print()
    info = pending_approval(agent, config)
    if info:
        print("\n>>> PAUSED, approval needed:", info["question"])
    return result

### Test 1 - add a task (custom tool)

In [ ]:
result = ask("Add a task: finish the marketing report by Friday")

### Test 2 - add another and list tasks

In [ ]:
result = ask("Also add: prepare slides for the exam. Then show me my task list.")

### Test 3 - weather (external API: Open-Meteo)

In [ ]:
result = ask("What is the weather in Tbilisi right now?")

### Test 4 - Wikipedia (external API)

In [ ]:
result = ask("Give me a short explanation of the Pomodoro Technique.")

### Test 5 - memory (same thread, no city mentioned)

In [ ]:
result = ask("Which city did I ask the weather for earlier?")

### Test 6 - complex multi-step query

In [ ]:
result = ask("Mark the first task as done, then tell me the weather in Batumi and whether I should bring an umbrella.")

## 5. Destructive action - APPROVED

Delete task -> agent pauses -> human says **yes** -> task is deleted. (New thread so the demo is clean.)

In [ ]:
from langgraph.types import Command

cfg_ok = {"configurable": {"thread_id": "demo-approve"}}
ask("Add a task: buy coffee for the team", thread_id="demo-approve")
print("\n===== NOW REQUEST THE DELETE =====\n")
ask("Delete the 'buy coffee for the team' task", thread_id="demo-approve")

In [ ]:
print("Tasks BEFORE approval:", TASKS)
print("\n===== HUMAN ANSWERS: yes =====\n")
result = agent.invoke(Command(resume="yes"), cfg_ok)
for m in result["messages"][-3:]:
    m.pretty_print()
print("Tasks AFTER approval:", TASKS)

## 6. Destructive action - DENIED

Delete task -> agent pauses -> human says **no** -> action is skipped.

In [ ]:
cfg_no = {"configurable": {"thread_id": "demo-deny"}}
ask("Add a task: submit the invoice to the client", thread_id="demo-deny")
print("\n===== NOW REQUEST THE DELETE =====\n")
ask("Delete the 'submit the invoice to the client' task", thread_id="demo-deny")

In [ ]:
print("\n===== HUMAN ANSWERS: no =====\n")
result = agent.invoke(Command(resume="no"), cfg_no)
for m in result["messages"][-3:]:
    m.pretty_print()
print("Tasks AFTER denial (the task must still be there):", TASKS)

## 7. Evaluation table

Run the notebook, then fill in **Pass?** and **Notes** honestly (a partial pass is still useful information).

| # | Query | Expected | Pass? | Notes |
|---|---|---|---|---|
| 1 | "Add a task: finish the marketing report by Friday" | `add_task` called, task #1 added | ⬜ | |
| 2 | "Also add: prepare slides... then show my list" | Two tasks added, list shows both | ⬜ | |
| 3 | "What is the weather in Tbilisi right now?" | `get_weather` called, temperature and wind returned | ⬜ | |
| 4 | "Explain the Pomodoro Technique" | `search_wikipedia` called, short summary returned | ⬜ | |
| 5 | "Which city did I ask the weather for earlier?" | Answers "Tbilisi" from memory, no tool needed | ⬜ | |
| 6 | "Mark the first task done, weather in Batumi, umbrella?" | `complete_task` + `get_weather` both used, sensible answer | ⬜ | |
| 7 | "Delete the task..." then **yes** | Approval requested, then task deleted | ⬜ | |
| 8 | "Delete the task..." then **no** | Approval requested, then action cancelled, task kept | ⬜ | |